# MacroChefAI — Data Pipeline

This notebook runs the end-to-end data processing pipeline:
1. Load the raw merged recipe CSV
2. Apply feature engineering (food tags, nutrition, Nutri-Score, medical flags)
3. Save processed parquet files to `processed_data/`
4. Fit and save TF-IDF and KNN models to `models/`

All functions are imported from `src/pipeline.py` — no logic is defined inline.

In [16]:
import os
import sys
import pandas as pd

# Ensure the project root is on the Python path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.pipeline import (
    # Paths
    RAW_DATA_PATH,
    PROCESSED_MODEL_PATH,
    PROCESSED_DISPLAY_PATH,
    TFIDF_PATH,
    INGREDIENT_MATRIX_PATH,
    KNN_MODEL_PATH,
    COMPACT_MODEL_PATH,
    # Pipeline functions
    load_and_process_recipe_csv,
    save_processed_recipe_data,
    fit_ingredient_tfidf,
    fit_macro_knn,
    # Loading helpers (for verification)
    load_processed_recipe_data,
)

print("Imports OK")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {RAW_DATA_PATH}")


Imports OK
Project root: /Users/ehtishamaziz/code/magedbekheet/macrochefai
Raw data: /Users/ehtishamaziz/code/magedbekheet/macrochefai/raw_data/merged_usable_cal_per_g.csv


## 1. Load and process raw data

Reads ~984 K rows from `raw_data/merged_usable_cal_per_g.csv`, standardises column names, parses ingredient and instruction lists, adds food type tags, computes per-100 g nutrition features, Nutri-Score, nutrient density score, medical risk flags, and macro labels.

In [17]:
df = load_and_process_recipe_csv(RAW_DATA_PATH)

print(f"Processed shape: {df.shape}")
display(df[[
    c for c in [
        "name", "calories", "protein", "fat", "carbs",
        "nutri_score_label", "nutrient_density_score",
    ] if c in df.columns
]].head(5))

Processed shape: (486662, 125)


,name,calories,protein,fat,carbs,nutri_score_label,nutrient_density_score
0,Cherry Streusel Cobbler,801.0,12.3,29.1,125.0,C,56.6
1,Reuben and Swiss Casserole Bake,664.4,31.0,45.3,33.9,D,56.0
2,Yam-Pecan Recipe,956.8,10.7,53.2,112.8,E,55.2
3,Tropical Orange Layer Cake,581.6,6.2,30.5,74.6,D,53.1
4,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,121.4,1.1,5.9,16.8,E,54.0


## 2. Save processed parquet files

Splits the dataframe into:
- **Model-ready** — numeric columns for the recommendation engine
- **Compact model** — lightweight subset for fast KNN queries
- **Display-ready** — presentation columns for the UI (names, descriptions, images, instructions)

In [18]:
save_processed_recipe_data(df)

# Verify output files
for path in [PROCESSED_MODEL_PATH, PROCESSED_DISPLAY_PATH, COMPACT_MODEL_PATH]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  ✅ {path}  ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {path}  MISSING")


Saved model file   -> /Users/ehtishamaziz/code/magedbekheet/macrochefai/processed_data/recipes_model_ready.parquet
Saved compact file -> /Users/ehtishamaziz/code/magedbekheet/macrochefai/processed_data/recipes_model_compact.parquet
Saved display file -> /Users/ehtishamaziz/code/magedbekheet/macrochefai/processed_data/recipes_display_ready.parquet
  ✅ /Users/ehtishamaziz/code/magedbekheet/macrochefai/processed_data/recipes_model_ready.parquet  (88.5 MB)
  ✅ /Users/ehtishamaziz/code/magedbekheet/macrochefai/processed_data/recipes_display_ready.parquet  (245.6 MB)
  ✅ /Users/ehtishamaziz/code/magedbekheet/macrochefai/processed_data/recipes_model_compact.parquet  (52.6 MB)


## 3. Fit and save TF-IDF model

Builds an ingredient-text TF-IDF vectoriser (unigrams + bigrams, max 30 K features) and saves the vectoriser and sparse matrix.

In [19]:
tfidf_vectorizer, ingredient_matrix = fit_ingredient_tfidf(df)

print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"Matrix shape: {ingredient_matrix.shape}")

Saved TF-IDF vectorizer -> /Users/ehtishamaziz/code/magedbekheet/macrochefai/models/tfidf_vectorizer.joblib
Saved ingredient matrix -> /Users/ehtishamaziz/code/magedbekheet/macrochefai/models/ingredient_matrix.npz
Vocabulary size: 30000
Matrix shape: (486662, 30000)


## 4. Fit and save KNN macro model

Trains a scikit-learn `NearestNeighbors` model on `[protein, fat, carbs]` scaled with `StandardScaler`, and saves the bundle (model + scaler + feature columns).

In [20]:
macro_knn_bundle = fit_macro_knn(
    df,
    feature_cols=("protein", "fat", "carbs"),
)

print(f"KNN trained on {macro_knn_bundle['model'].n_samples_fit_} samples")
print(f"Feature columns: {macro_knn_bundle['feature_cols']}")

Saved: /Users/ehtishamaziz/code/magedbekheet/macrochefai/models/macro_knn.joblib
KNN trained on 486662 samples
Feature columns: ['protein', 'fat', 'carbs']


## 5. Verify: reload processed data

Quick smoke test — load the saved parquets and confirm dimensions and key columns.

In [21]:
df_check = load_processed_recipe_data(include_display=True)

print(f"Reloaded shape: {df_check.shape}")
print(f"Columns ({len(df_check.columns)}): {list(df_check.columns[:20])} ...")

# Spot check a recipe
sample = df_check.sample(1).iloc[0]
print(f"\nSample recipe: {sample.get('name', 'N/A')}")
print(f"  Calories: {sample.get('calories', 'N/A')}")
print(f"  Nutri-Score: {sample.get('nutri_score_label', 'N/A')}")
print(f"  Medical risk: {sample.get('medical_risk_level', 'N/A')} — {sample.get('medical_risk_reason', 'N/A')}")

Reloaded shape: (486662, 85)
Columns (85): ['recipe_id', 'name', 'final_name', 'RecipeCategory', 'dd_meal_type', 'image_url', 'ingredients_text', 'cook_time', 'servings', 'serving_g', 'calories', 'protein', 'fat', 'sat_fat', 'carbs', 'fiber', 'sugar', 'cholest', 'sodium', 'cal_per_g'] ...

Sample recipe: Mom's Potato Casserole
  Calories: 529.6
  Nutri-Score: C
  Medical risk: moderate — high saturated fat, high carbohydrates


## 6. Demo recommendation

Test the pipeline by running a sample recommendation query using saved models.

In [22]:
from src.nutrition import calculate_bmi, calculate_bmr, calculate_tdee, adjust_calories, calculate_macros

user_profile = {
    "age": 30,
    "weight": 70,
    "height": 170,
    "sex": "male",
    "activity_level": "moderate",
    "goal": "weight_loss",
}

bmr = calculate_bmr(user_profile["weight"], user_profile["height"], user_profile["age"], user_profile["sex"])
tdee = calculate_tdee(bmr, user_profile["activity_level"])
target_cal = adjust_calories(tdee, user_profile["goal"], user_profile["sex"])
macros = calculate_macros(target_cal, user_profile["goal"], user_profile["weight"])

print("User targets:")
print(f"  BMR: {bmr:.0f} kcal")
print(f"  TDEE: {tdee:.0f} kcal")
print(f"  Daily target: {target_cal:.0f} kcal")
print(f"  Per-meal (3 meals): {target_cal/3:.0f} kcal")
print(f"  Protein: {macros['protein_g']:.1f} g")
print(f"  Fat: {macros['fat_g']:.1f} g")
print(f"  Carbs: {macros['carbs_g']:.1f} g")


User targets:
  BMR: 1619 kcal
  TDEE: 2510 kcal
  Daily target: 2008 kcal
  Per-meal (3 meals): 669 kcal
  Protein: 126.0 g
  Fat: 56.0 g
  Carbs: 250.0 g


---

✅ **Pipeline complete.** The processed data and models are ready for the app.

```bash
streamlit run streamlit_app.py
```